# Week 9: Operational Machine Learning — Predictive Maintenance for Pipeline Pumps

**Capstone context:** Predictive Maintenance for KPC Pipeline Pump Infrastructure  
**Author:** Lameck Mugo  
**Target variable:** `failure_within_7_days`

## Operational problem
Unplanned pump failures can interrupt fuel throughput, delay deliveries, increase emergency repair costs, and create safety-critical operating conditions. The operational objective is therefore to identify pumps at elevated risk **before failure**, giving maintenance teams time to inspect or intervene.

The capstone proposal states two important operational constraints:

- Target at least **80% recall** for near-failure detection.
- Keep the **false-positive rate below 5%** to protect technician trust.

For this Week 9 implementation, the primary dataset is a **physics-informed synthetic SCADA dataset** because real internal KPC sensor and maintenance logs are not publicly available. The synthetic data models the telemetry named in the capstone proposal: vibration, temperature, motor current, pressure, flow, load, pump age, and maintenance state.

The UCI AI4I 2020 Predictive Maintenance Dataset can later be used as a secondary benchmark for feature-engineering sanity checks. It contains 10,000 observations and a `Machine failure` target. The official UCI dataset page documents its variables and failure modes.

**Important methodological note:** This notebook does not claim that the synthetic model is ready for production. Synthetic data can be useful for prototyping the workflow, but production deployment requires validation against real KPC operational and maintenance data.


## 1. Environment and imports

Install the required packages once in the VS Code terminal if necessary:

```bash
pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost shap jupyter
```


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, roc_auc_score,
    RocCurveDisplay
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from xgboost import XGBClassifier
import shap

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## 2. Generate physics-informed synthetic SCADA data

The capstone proposal explicitly states that the primary data source is physics-informed synthetic SCADA data because KPC internal logs are unavailable.

The relationships below are deliberately simplified. They encode the operational idea that failure risk can increase when several warning signals deteriorate together, such as vibration, bearing temperature, motor current, pressure deviation, pump age, and overdue maintenance.

This gives us an operational classification problem with an intentionally imbalanced target.


In [ ]:
# Number of synthetic SCADA observations
n = 12_000

station_id = rng.choice([f"Station_{i}" for i in range(1, 11)], size=n)
pump_age_years = rng.uniform(0.5, 15, size=n)
load_factor = rng.uniform(0.55, 1.05, size=n)
ambient_temp_c = rng.normal(29, 4, size=n)

# Vibration rises with degradation and operating load.
vibration_mm_s = np.clip(
    rng.gamma(shape=2, scale=1.1, size=n)
    + 0.22 * pump_age_years
    + 0.9 * (load_factor - 0.75),
    0, None
)

# Bearing temperature responds to age, load and vibration.
bearing_temp_c = (
    55
    + 0.8 * pump_age_years
    + 15 * (load_factor - 0.6)
    + 5 * vibration_mm_s
    + rng.normal(0, 4, size=n)
)

# Motor current rises with load and mechanical stress.
motor_current_a = (
    140
    + 85 * load_factor
    + 3 * pump_age_years
    + 12 * vibration_mm_s
    + rng.normal(0, 8, size=n)
)

# Deviations represent unstable operating behaviour.
pressure_deviation_bar = np.abs(
    rng.normal(0, 3, size=n)
    + 2.2 * (load_factor - 0.8)
    + 0.9 * vibration_mm_s
)

flow_deviation_pct = np.abs(
    rng.normal(0, 4, size=n)
    + 2.0 * (load_factor - 0.8)
    + 0.7 * vibration_mm_s
)

days_since_maintenance = np.clip(
    rng.gamma(shape=2, scale=25, size=n)
    + 4 * pump_age_years
    + rng.normal(0, 5, size=n),
    0, None
)

operating_hours_since_maintenance = days_since_maintenance * 24

station_effect_map = {
    f"Station_{i}": value
    for i, value in enumerate(np.linspace(-0.25, 0.25, 10), start=1)
}
station_effect = pd.Series(station_id).map(station_effect_map).to_numpy()

# Latent near-failure risk.
risk_score = (
    -16
    + 1.0 * vibration_mm_s
    + 0.055 * (bearing_temp_c - 70)
    + 0.035 * (motor_current_a - 200)
    + 0.17 * pressure_deviation_bar
    + 0.10 * flow_deviation_pct
    + 0.10 * pump_age_years
    + 0.025 * (days_since_maintenance - 40)
    + station_effect
)

failure_probability = 1 / (1 + np.exp(-risk_score))
failure_within_7_days = rng.binomial(1, failure_probability)

df = pd.DataFrame({
    "station_id": station_id,
    "pump_age_years": pump_age_years,
    "load_factor": load_factor,
    "ambient_temp_c": ambient_temp_c,
    "vibration_mm_s": vibration_mm_s,
    "bearing_temp_c": bearing_temp_c,
    "motor_current_a": motor_current_a,
    "pressure_deviation_bar": pressure_deviation_bar,
    "flow_deviation_pct": flow_deviation_pct,
    "days_since_maintenance": days_since_maintenance,
    "operating_hours_since_maintenance": operating_hours_since_maintenance,
    "failure_within_7_days": failure_within_7_days
})

print("Dataset shape:", df.shape)
display(df.head())


## 3. Problem definition and class-imbalance check

The target is `failure_within_7_days`:

- `1` = pump is predicted to experience a near-failure event within the operational intervention window.
- `0` = no near-failure event within that window.

Accuracy is not sufficient here. If failures are relatively rare, a model could achieve high accuracy simply by predicting "no failure" most of the time. Operationally, the model must balance:

- **Recall:** How many genuinely risky pumps were caught?
- **Precision:** Of the pumps flagged, how many were genuinely risky?
- **False-positive rate:** How many healthy pumps were unnecessarily flagged?
- **F1-score:** Balance between precision and recall.
- **ROC-AUC:** Overall ability to rank risky cases above non-risky cases across thresholds.


In [ ]:
target = "failure_within_7_days"

class_counts = df[target].value_counts().sort_index()
class_pct = df[target].value_counts(normalize=True).sort_index() * 100

class_distribution = pd.DataFrame({
    "count": class_counts,
    "percentage": class_pct
})
class_distribution.index = ["No near-failure (0)", "Near-failure (1)"]
display(class_distribution)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target)
plt.title("Target Class Distribution")
plt.xlabel("Failure within 7 days")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()

imbalance_ratio = class_counts[0] / class_counts[1]
print(f"Majority/minority ratio: {imbalance_ratio:.2f}:1")


## 4. Train/test split and preprocessing

The test set is held out until final model evaluation. Cross-validation is performed only on the training data.

Categorical station identifiers are one-hot encoded. Numeric columns are median-imputed. The dataset is generated without missing values, but the imputation step keeps the pipeline reusable if missing telemetry appears later.

For imbalance handling:

- **Random Forest:** `class_weight="balanced"` is used.
- **XGBoost:** **SMOTE is applied inside the cross-validation pipeline**, preventing information leakage from synthetic minority examples into validation folds.

SMOTE is intentionally placed inside the pipeline rather than being applied before the split or before cross-validation.


In [ ]:
X = df.drop(columns=target)
y = df[target]

categorical_features = ["station_id"]
numeric_features = [c for c in X.columns if c not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Training observations:", X_train.shape[0])
print("Test observations:", X_test.shape[0])
print("Training failure prevalence:", f"{y_train.mean():.2%}")
print("Test failure prevalence:", f"{y_test.mean():.2%}")


## 5. Train two operational models with cross-validation

Two ensemble models are compared:

1. **Random Forest** — robust non-linear baseline with class weighting.
2. **XGBoost** — gradient-boosted trees, useful for complex interactions among operational sensor signals.

Five-fold stratified cross-validation is used so each fold preserves approximately the same failure prevalence.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.90,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

xgb_pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
    ("model", xgb_model)
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

rf_cv = cross_validate(
    rf_pipeline, X_train, y_train,
    cv=cv, scoring=scoring, n_jobs=-1
)

# Use one process because XGBoost itself can parallelize.
xgb_cv = cross_validate(
    xgb_pipeline, X_train, y_train,
    cv=cv, scoring=scoring, n_jobs=1
)

cv_results = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost + SMOTE"],
    "ROC-AUC": [
        rf_cv["test_roc_auc"].mean(),
        xgb_cv["test_roc_auc"].mean()
    ],
    "Precision": [
        rf_cv["test_precision"].mean(),
        xgb_cv["test_precision"].mean()
    ],
    "Recall": [
        rf_cv["test_recall"].mean(),
        xgb_cv["test_recall"].mean()
    ],
    "F1": [
        rf_cv["test_f1"].mean(),
        xgb_cv["test_f1"].mean()
    ]
}).set_index("Model")

display(cv_results.round(3))


## 6. Final model selection and threshold tuning

The final operating threshold should reflect the capstone's operational targets, rather than automatically defaulting to 0.50.

Here, XGBoost is selected for threshold analysis. The threshold is chosen by searching for an operating point that:

- achieves recall of at least **80%**, and
- keeps the false-positive rate at or below **5%**.

This is important because the probability threshold determines the operational trade-off between missed failures and unnecessary inspections.


In [ ]:
# Fit both models on the complete training set.
rf_pipeline.fit(X_train, y_train)
xgb_pipeline.fit(X_train, y_train)

rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]
xgb_prob = xgb_pipeline.predict_proba(X_test)[:, 1]

threshold_rows = []

for threshold in np.arange(0.05, 0.96, 0.01):
    pred = (xgb_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "false_positive_rate": fp / (fp + tn)
    })

threshold_results = pd.DataFrame(threshold_rows)

feasible_thresholds = threshold_results[
    (threshold_results["recall"] >= 0.80)
    & (threshold_results["false_positive_rate"] <= 0.05)
].sort_values(["f1", "recall"], ascending=False)

if feasible_thresholds.empty:
    print("No threshold met both operational constraints exactly.")
    selected_threshold = 0.50
else:
    selected_threshold = float(feasible_thresholds.iloc[0]["threshold"])

print("Selected operating threshold:", selected_threshold)
display(feasible_thresholds.head(10).round(3))


## 7. Final evaluation

The confusion matrix is especially important operationally:

- **True Positive:** At-risk pump correctly flagged.
- **False Negative:** At-risk pump missed. This is the most safety-sensitive error because maintenance may not intervene in time.
- **False Positive:** Healthy pump flagged. This consumes technician time and may reduce trust in alerts.
- **True Negative:** Healthy pump correctly not escalated.

Accuracy is deliberately not used as the primary decision metric.


In [ ]:
rf_pred = (rf_prob >= 0.50).astype(int)
xgb_pred = (xgb_prob >= selected_threshold).astype(int)

def evaluate_model(name, y_true, y_pred, probabilities):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Model": name,
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "False Positive Rate": fp / (fp + tn),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

final_results = pd.DataFrame([
    evaluate_model("Random Forest @ 0.50", y_test, rf_pred, rf_prob),
    evaluate_model(
        f"XGBoost + SMOTE @ {selected_threshold:.2f}",
        y_test, xgb_pred, xgb_prob
    )
])

display(final_results.round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, rf_pred,
    display_labels=["No failure", "Near-failure"],
    ax=axes[0], values_format="d"
)
axes[0].set_title("Random Forest Confusion Matrix")

ConfusionMatrixDisplay.from_predictions(
    y_test, xgb_pred,
    display_labels=["No failure", "Near-failure"],
    ax=axes[1], values_format="d"
)
axes[1].set_title(f"XGBoost + SMOTE @ threshold {selected_threshold:.2f}")

plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, rf_prob, name="Random Forest")
RocCurveDisplay.from_predictions(y_test, xgb_prob, name="XGBoost + SMOTE")
plt.title("ROC Curves")
plt.tight_layout()
plt.show()


## 8. Optional bonus: K-Means clustering

Clustering does not predict failure directly. Instead, it can help the operations team identify groups of operating conditions or pump states.

Only numeric operational features are used and standardized because K-Means is distance-based. The clusters are then profiled by average telemetry and observed near-failure rate.


In [ ]:
cluster_features = [
    "vibration_mm_s",
    "bearing_temp_c",
    "motor_current_a",
    "pressure_deviation_bar",
    "flow_deviation_pct",
    "pump_age_years",
    "days_since_maintenance"
]

scaler = StandardScaler()
X_cluster = scaler.fit_transform(df[cluster_features])

# Three segments: relatively stable, intermediate monitoring, and higher-stress operation.
kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=20)
df["cluster"] = kmeans.fit_predict(X_cluster)

cluster_profile = (
    df.groupby("cluster")[cluster_features + [target]]
      .mean()
      .sort_values(target, ascending=False)
)

print("Cluster profiles:")
display(cluster_profile.round(2))

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df.sample(min(3000, len(df)), random_state=RANDOM_STATE),
    x="vibration_mm_s",
    y="bearing_temp_c",
    hue="cluster",
    alpha=0.6
)
plt.title("Operational Segments from K-Means")
plt.tight_layout()
plt.show()


## 9. Feature importance

Tree-based feature importance gives a global view of which variables the XGBoost model used most when separating near-failure from non-failure observations.

This should not be interpreted as causation. A high importance score means the model found the feature useful for prediction; it does not prove that changing the feature alone causes failure.


In [ ]:
xgb_fitted_model = xgb_pipeline.named_steps["model"]
feature_names = xgb_pipeline.named_steps["preprocessor"].get_feature_names_out()

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_fitted_model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(15))

top_n = 12
plot_data = feature_importance.head(top_n).sort_values("importance")

plt.figure(figsize=(9, 6))
plt.barh(plot_data["feature"], plot_data["importance"])
plt.xlabel("Feature importance")
plt.title("Top XGBoost Feature Importances")
plt.tight_layout()
plt.show()


## 10. SHAP explainability

SHAP provides two complementary explanations:

1. **Global summary plot:** which features generally move predictions toward higher or lower failure risk.
2. **Local explanation:** why one specific pump observation was assigned a high risk score.

For the video deliverable, select a high-risk test observation and show its SHAP force plot. The narration should explain the prediction in operational language, for example:

> "Here is a case the model flagged as High Risk. The vibration level pushed the prediction upward most strongly, while bearing temperature and motor current added further evidence of mechanical stress."


In [ ]:
# Transform test data using the fitted preprocessing stage.
X_test_transformed = xgb_pipeline.named_steps["preprocessor"].transform(X_test)

# Explain a representative sample for the global SHAP summary.
sample_size = min(500, X_test_transformed.shape[0])
sample_idx = rng.choice(
    X_test_transformed.shape[0],
    size=sample_size,
    replace=False
)

explainer = shap.TreeExplainer(xgb_fitted_model)
shap_values = explainer.shap_values(X_test_transformed[sample_idx])

# Global explanation
shap.summary_plot(
    shap_values,
    X_test_transformed[sample_idx],
    feature_names=feature_names,
    show=True
)


In [ ]:
# Choose the highest-risk case on the held-out test set.
high_risk_index = int(np.argmax(xgb_prob))
high_risk_probability = float(xgb_prob[high_risk_index])

print("Selected high-risk test observation:", high_risk_index)
print(f"Predicted near-failure probability: {high_risk_probability:.2%}")
display(X_test.iloc[[high_risk_index]])

# Local SHAP values for the selected case.
local_shap_values = explainer.shap_values(
    X_test_transformed[[high_risk_index]]
)

shap.force_plot(
    explainer.expected_value,
    local_shap_values[0],
    X_test_transformed[high_risk_index],
    feature_names=feature_names,
    matplotlib=True,
    show=True
)


## 11. Operational interpretation and limitations

### How operations should use the model
The model should be treated as a **risk-prioritization tool**, not an automatic maintenance decision-maker.

A practical workflow could be:

1. Model scores current pump telemetry.
2. Pumps above the selected threshold enter a maintenance review queue.
3. An engineer or technician checks sensor validity, recent maintenance history, process conditions, and safety procedures.
4. The team decides whether inspection, monitoring, load reduction, or maintenance is justified.
5. Actual outcomes are fed back into future model validation.

### Main limitations
- The primary dataset is synthetic and therefore cannot establish real-world performance.
- Sensor drift, missing telemetry, and changing operating regimes may reduce performance.
- The chosen threshold is an operational policy choice and may need adjustment.
- False negatives can miss genuinely deteriorating pumps.
- False positives can create unnecessary inspections and alert fatigue.
- Feature importance and SHAP explain model behaviour; they do not prove physical causation.

### Final model choice
Choose the model based on the full operational trade-off rather than a single score. In this capstone context, the model meeting the recall target while keeping the false-positive rate within the operational tolerance is preferred.


## 12. Reproducibility

Recommended project structure:

```text
week9/
├── week9_operational_ml.ipynb
├── requirements.txt
├── Week9_Model_Explainer_LameckMugo.pdf
├── Week9_Video_LameckMugo.mp4
└── capstone_week9_update.md
```

Suggested `requirements.txt`:

```text
pandas
numpy
matplotlib
seaborn
scikit-learn
imbalanced-learn
xgboost
shap
jupyter
```
